# 택스트 특징 추출 (TF-IDF)
 
### 1. 샘플 코퍼스로 연습

In [3]:
# 분석할 문장들을 담고 있는 리스트 (학습용 데이터셋)
sample_corpus = [
    '자연어처리 강의를 시작하겠습니다.',
    '자연어처리는 재미있습니다.',
    '밥을 먹고 강의를 듣고 있습니다.',
    '이번 자연어처리 강의는 한국어 자연어처리입니다.'
]

In [4]:
# KoNLPy 라이브러리에서 Okt(Open Korean Text) 형태소 분석기 클래스를 불러오기
from konlpy.tag import Okt
# 문장을 입력받아 토큰(명사 리스트)으로 변환해주는 사용자 정의 함수를 선언
def my_tokenizer(text):
    # 텍스트가 유효하지 않을 경우(NaN 등)를 대비해 str() 처리
    stop_words = ['만', '들을', '하고', '없다', '에', '의', '가', '이', '은', '는', '영화', '정말', '진짜', '적', '함']
    tokens = Okt().pos(text)
    # 명사, 동사, 형용사 추출 + 불용어 제거 + 2글자 이상 단어만 선택
    return [word for word, pos in tokens 
            if pos in ['Noun', 'Verb', 'Adjective'] 
            and word not in stop_words 
            and len(word) > 1]

In [5]:
# Tfidvectorizer 객체 생성
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer(tokenizer=my_tokenizer)

# 특징 집합과 관련 데이터 모델을 생성
vectorizer.fit(sample_corpus)
print(vectorizer.get_feature_names_out())
# 특징 벡터 추출
sample_dtm = vectorizer.transform(sample_corpus)
type(sample_dtm)
print(sample_dtm.toarray())

['강의' '듣고' '먹고' '시작' '이번' '입니다' '있습니다' '자연어' '재미있습니다' '처리' '하겠습니다' '한국어']
[[0.3555803  0.         0.         0.55708525 0.         0.
  0.         0.3555803  0.         0.3555803  0.55708525 0.        ]
 [0.         0.         0.         0.         0.         0.
  0.         0.47380449 0.74230628 0.47380449 0.         0.        ]
 [0.34578314 0.5417361  0.5417361  0.         0.         0.
  0.5417361  0.         0.         0.         0.         0.        ]
 [0.24720702 0.         0.         0.         0.38729756 0.38729756
  0.         0.49441404 0.         0.49441404 0.         0.38729756]]


c:\Users\user\anaconda3\envs\textmine26\lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


### 2. 다음 영화 리뷰로 연습

#### 2-1. 파일 경로 정리 및 리뷰 전처리

In [ ]:
import os
import pandas as pd
from konlpy.tag import Okt
from sklearn.feature_extraction.text import TfidfVectorizer

# 1. 환경 설정 및 데이터 로드
# KoNLPy 사용을 위한 Java 경로 설정 (본인 환경에 맞게 확인 필요)
os.environ['JAVA_HOME'] = r'C:\Program Files\Java\jdk-17' 

# 형태소 분석기 객체를 함수 밖에서 생성하여 속도 최적화
okt = Okt()

# 사용자 정의 토크나이저2 함수 (처음 실습 한 함수와 단어 충돌 방지용으로 2로 식별)
def my_tokenizer2(text):
    # 명사(nouns)만 추출하여 리스트로 반환
    return okt.nouns(str(text)) 

# 실제 본인 컴퓨터의 파일 경로로 데이터 로드
datafile = r'C:\Users\user\Desktop\textmine_26\data\daum_movie_review.csv'
data_df = pd.read_csv(datafile)

# 2. TfidfVectorizer 객체 생성 및 학습
# tokenizer=my_tokenizer: 명사 추출 함수 연결
# max_features=100: 중요도 상위 100개 단어만 추출
# token_pattern=None: 사용자 정의 토크나이저 사용 시 발생하는 워닝 방지
vectorizer = TfidfVectorizer(tokenizer=my_tokenizer2, max_features=100, token_pattern=None)


In [8]:
# 리뷰 텍스트 원문을 학습 (내부에서 my_tokenizer가 자동으로 명사를 추출함)
vectorizer.fit(data_df['review'])

# 추출된 상위 100개의 핵심 단어(특징 이름) 목록 출력
print("추출된 특징 단어 목록:")
print(vectorizer.get_feature_names_out())


추출된 특징 단어 목록:
['가슴' '가족' '감독' '감동' '거' '걸' '것' '공포' '공포영화' '관객' '광주' '그' '그냥' '기대' '꼭'
 '끝' '나' '내' '내용' '노스' '눈' '눈물' '느낌' '다시' '더' '도' '돈' '듯' '때' '때문' '또'
 '마동석' '마블' '마음' '마지막' '만' '말' '모두' '몰입' '뭐' '배우' '번' '별로' '보고' '볼' '부분'
 '분' '사람' '사랑' '생각' '송강호' '수' '수준' '스토리' '시간' '신' '신파' '안' '액션' '역사' '역시'
 '연기' '연출' '영화' '완전' '왜' '우리' '원작' '웹툰' '윤계상' '음악' '이' '이야기' '이해' '인생' '임'
 '작품' '장면' '재미' '저' '점' '정도' '정말' '조금' '좀' '중' '중간' '진짜' '차태현' '처음' '최고'
 '추천' '택시' '편' '평점' '하나' '한국' '한번' '함' '현실']


In [9]:
# 3. 특징 벡터 추출 (DTM 생성)
# 리뷰 원문을 수치 행렬로 변환
sample_dtm = vectorizer.transform(data_df['review'])

# 희소 행렬을 일반 배열(Array) 형태로 변환하여 출력
print("\nTF-IDF 수치 행렬 (상위 일부):")
print(sample_dtm.toarray())


TF-IDF 수치 행렬 (상위 일부):
[[0.         0.         0.         ... 0.         0.         0.        ]
 [0.         0.         0.         ... 0.         0.         0.        ]
 [0.         0.         0.         ... 0.         0.         0.        ]
 ...
 [0.         0.75428951 0.         ... 0.         0.         0.        ]
 [0.         0.         0.         ... 0.         0.         0.        ]
 [0.         0.         0.         ... 0.         0.         0.        ]]
